In [1]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from scipy.signal import argrelextrema
#from scipy.stats import ttest_ind, ttest_1samp, fisher_exact
#import statsmodels.api as sm

In [2]:
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['axes.labelsize'] = 'small'
matplotlib.rcParams['axes.linewidth'] = 1
matplotlib.rcParams['lines.markersize'] = 2
matplotlib.rcParams['xtick.major.width'] = 1
matplotlib.rcParams['ytick.major.width'] = 1
matplotlib.rcParams['xtick.labelsize'] = 'x-small'
matplotlib.rcParams['ytick.labelsize'] = 'x-small'
matplotlib.rcParams['legend.fontsize'] = 'x-small'
matplotlib.rcParams['figure.dpi'] = 600

In [3]:
# Read birthdating data
birthdating_df = pd.read_excel('./Dataset_S1.xlsx', sheet_name='Data')

# Find cutoff using Maximum KGG peptides log2 ratio +- MG132
data = birthdating_df['Maximum KGG peptides log2 ratio +- MG132'].to_numpy()
data = data[~np.isnan(data)]

nbins = 50

p, e = np.histogram(data, bins=nbins, density=True)
idx_list = argrelextrema(p, np.less)[0]
cutoff = (e[idx_list] + e[idx_list+1])/2
cutoff = np.min(cutoff[cutoff > 0])

# Label proteins
birthdating_labelled_df = pd.DataFrame({
    'Uniprot': birthdating_df['Uniprot'],
    'KGG peptides detected?': birthdating_df['KGG peptides detected'].apply(lambda x: 'Y' if x else 'N'),
    'KGG increase w MG132?': birthdating_df['Maximum KGG peptides log2 ratio +- MG132'].apply(lambda x: 'Y' if pd.notna(x) and x > cutoff else ('N' if pd.notna(x) else np.nan))
})

# Find cutoff using Log2 ratio of youngest KGG peptide vs. median of unmodified peptides
data = birthdating_df['Log2 ratio of youngest KGG peptide vs. median of unmodified peptides'].to_numpy()
data = data[~np.isnan(data)]

nbins = 50

p, e = np.histogram(data, bins=nbins, density=True)
idx_list = argrelextrema(p, np.less)[0]
cutoff = (e[idx_list] + e[idx_list+1])/2
cutoff = np.max(cutoff[cutoff < 0])
print('Max Cutoff = %.4f'%cutoff)

# Label proteins
birthdating_labelled_df['KGG young?'] = birthdating_df['Log2 ratio of youngest KGG peptide vs. median of unmodified peptides'].apply(lambda x: 'Y' if pd.notna(x) and x < cutoff else ('N' if pd.notna(x) else np.nan))

# Label proteins
cutoff = 0.25 # day
birthdating_labelled_1d_df = birthdating_labelled_df[['Uniprot', 'KGG peptides detected?', 'KGG increase w MG132?', 'KGG young?']].copy()
birthdating_labelled_1d_df['Youngest KGG age < '+str(cutoff)+' day?'] = birthdating_df['Minimum age of KGG peptides'].apply(lambda x: 'Y' if pd.notna(x) and x < cutoff else ('N' if pd.notna(x) else np.nan))

# Criteria #1: 
# YU - "KGG peptides detected?" = Y
#      & "KGG increase w MG132?" = Y
#      & "KGG young?" = Y
#      & "Youngest KGG age < 1 day?" = Y;
# N - "KGG peptides detected?" = N;
# NA - Others
labels = []
for index, row in birthdating_labelled_1d_df.iterrows():
    if row['KGG peptides detected?'] == 'Y' \
        and row['KGG increase w MG132?'] == 'Y' \
        and row['KGG young?'] == 'Y' \
        and row['Youngest KGG age < '+str(cutoff)+' day?'] == 'Y':
        label = 'YU'
    elif row['KGG peptides detected?'] == 'N':
        label = 'N'
    else:
        label = 'NA'
    labels.append(label)
birthdating_labelled_1d_df['Label #1'] = labels

display(birthdating_labelled_1d_df)

Max Cutoff = -1.8510


,Uniprot,KGG peptides detected?,KGG increase w MG132?,KGG young?,Youngest KGG age < 0.25 day?,Label #1
0,P60709,Y,Y,Y,Y,YU
1,P78417,Y,Y,Y,Y,YU
2,P14618,Y,Y,Y,Y,YU
3,P52272,Y,Y,Y,N,NA
4,P0DMV8,Y,Y,Y,Y,YU
...,...,...,...,...,...,...
6445,Q9Y6N1,N,NaN,NaN,NaN,N
6446,Q9Y6R0,N,NaN,NaN,NaN,N
6447,Q9Y6R4,N,NaN,NaN,NaN,N
6448,Q9Y6W5,N,NaN,NaN,NaN,N


In [4]:
# Load knotted proteins
knotted_prot_df = pd.read_table('./AF4_knots.dat')
knotted_prot_df['Uniprot'] = knotted_prot_df['Uniprot'].apply(lambda x: x.split('-')[0])
human_knotted_prot_df = knotted_prot_df.loc[knotted_prot_df['Organism']=='Human']
human_knotted_prot_df['Knotted'] = 1
display(human_knotted_prot_df)

/var/folders/hk/qv3d0x814pl701svvdt596x40000gn/T/ipykernel_77838/1925132482.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  human_knotted_prot_df['Knotted'] = 1


,#Knot_type,Category,Uniprot,Organism,pLDDT_knotcore,Protein_name,Knotted
1495,Unknown,AF4,Q7Z5N4,Human,75.4,Protein sidekick-1,1
2459,9_21,AF4,B4E1Z4,Human,71.1,Complement factor B (EC 3.4.21.47) (C3/C5 conv...,1
5355,6_3,AF4,O00534,Human,89.9,von Willebrand factor A domain-containing prot...,1
6394,6_3,AF4,A8K6N3,Human,87.2,"cDNA FLJ76886, highly similar to Homo sapiens ...",1
10020,5_2,AF4,P15374,Human,95.4,Ubiquitin carboxyl-terminal hydrolase isozyme ...,1
...,...,...,...,...,...,...,...
678998,3_1,AF4,B7Z3H4,Human,43.5,Beta-transducin repeat containing isoform 4 (B...,1
679396,3_1,AF4,Q53F78,Human,41.0,Beclin-1,1
679439,3_1,AF4,Q7Z6L1,Human,40.6,Tectonin beta-propeller repeat-containing prot...,1
679647,3_1,AF4,Q9H8X9,Human,39.2,Palmitoyltransferase ZDHHC11,1


In [5]:
# Load membrane protein uniprot IDs
with open('./uniprotkb_human_keywords_membrane_2025_03_19.fasta') as f:
    lines = f.readlines()
mem_prot_uid_list = []
for line in lines:
    line = line.strip()
    if line.startswith('>'):
        uid = line.split('|')[1]
        mem_prot_uid_list.append([uid, 'Y'])
mem_prot_df = pd.DataFrame(mem_prot_uid_list, columns=['Uniprot', 'Membrane?'])
display(mem_prot_df)

,Uniprot,Membrane?
0,A0A087X0K9,Y
1,A0A087X1C5,Y
2,A0A0A0MR25,Y
3,A0A0B4J2F0,Y
4,A0A0C3SFZ9,Y
...,...,...
18636,X6RF05,Y
18637,X6RI56,Y
18638,X6RIG5,Y
18639,X6RKN2,Y


In [6]:
# Function to merge lists while keeping NaN intact
def merge_CCBond(series):
    data = []
    for item in series.dropna():
        if isinstance(item, bool):
            data.append(item)
        elif item == 'True':
            data.append(True)
        elif item == 'False':
            data.append(False)
        elif item == '1.0':
            data.append(True)
        elif item == '0.0':
            data.append(False)

    if len(data) > 0:
        if np.any(data): # At least one covalent lasso => excluded
            return np.nan
        else:
            return 1
    else:
        return 0

# Parse protein lengths
with open('./UP000005640_9606.fasta') as f:
    lines = f.readlines()
prot_len_dict = {}
for line in lines:
    line = line.strip()
    if line.startswith('>'):
        uid = line.split('|')[1]
        prot_len_dict[uid] = 0
    else:
        prot_len_dict[uid] += len(line)
prot_len_df = pd.DataFrame(list(prot_len_dict.items()), columns=['gene', 'Protein length'])

# AF2 structures
pLDDT_cutoff = 85
AF_ent_data = pd.read_csv('./Human_AF_combined_20250614.csv', sep='|')
AF_ent_data = AF_ent_data[['gene','CCBond']].groupby('gene', as_index=False).agg({'CCBond': merge_CCBond})
print('Total %d AF2 proteins'%AF_ent_data.shape[0])
# AF_ent_data = AF_ent_data.dropna(subset='CCBond')
# print('%d AF2 proteins DO NOT have covalent lassos'%AF_ent_data.shape[0])
AF_ent_data = AF_ent_data.merge(prot_len_df, on='gene', how='inner')
AF_ent_data = AF_ent_data.merge(pd.read_csv('./Human_Avg_pLDDTs.csv')[['gene','<pLDDT>']], on='gene', how='inner')
print('%d AF2 proteins left with protein legnth available'%AF_ent_data.shape[0])
AF_ent_data = AF_ent_data.loc[AF_ent_data['<pLDDT>'] >= pLDDT_cutoff]
print('%d AF2 proteins with <pLDDT> >= %d'%(AF_ent_data.shape[0], pLDDT_cutoff))
AF_ent_data.rename(columns={'gene':'Uniprot',
                            'CCBond':'Entanglement'}, inplace=True)
AF_ent_data = AF_ent_data.merge(human_knotted_prot_df[['Uniprot', 'Knotted']], on='Uniprot', how='left')
AF_ent_data['Knotted'] = AF_ent_data['Knotted'].fillna(0)

display(AF_ent_data)

Total 20588 AF2 proteins
20367 AF2 proteins left with protein legnth available
5856 AF2 proteins with <pLDDT> >= 85


,Uniprot,Entanglement,Protein length,<pLDDT>,Knotted
0,A0A075B6H5,0.0,111,85.970923,0.0
1,A0A075B6H7,0.0,116,90.421034,0.0
2,A0A075B6H8,0.0,117,91.096667,0.0
3,A0A075B6H9,0.0,119,89.549328,0.0
4,A0A075B6I0,0.0,122,89.309672,0.0
...,...,...,...,...,...
5851,S4R3C0,1.0,116,89.570259,0.0
5852,S4R3P1,0.0,24,87.620833,0.0
5853,S4R3Y5,0.0,24,87.637083,0.0
5854,U3KPV4,1.0,340,90.760618,0.0


In [7]:
print('Total number of proteins detected: %d'%(len(birthdating_labelled_1d_df)))
num_membrane_prot = len(birthdating_labelled_1d_df.merge(mem_prot_df, on='Uniprot', how='inner'))
print('%d are membrane proteins according to UniProt'%(num_membrane_prot))
birthdating_AF_ent_df = birthdating_labelled_1d_df.merge(AF_ent_data, on='Uniprot', how='inner')
print('%d have high quality AlphaFold structures'%(len(birthdating_AF_ent_df)))
print('Amongst high quality structure, %d have one or more native entanglements'%(len(birthdating_AF_ent_df[birthdating_AF_ent_df['Entanglement']==1])))
print('%d have covalent lassos'%(len(birthdating_AF_ent_df[pd.isna(birthdating_AF_ent_df['Entanglement'])])))
print('%d have knots'%(len(birthdating_AF_ent_df[birthdating_AF_ent_df['Knotted']==1])))

Ubq_protein_df = birthdating_labelled_1d_df[(birthdating_labelled_1d_df['Youngest KGG age < 0.25 day?'] == 'Y')]
print('Total number of ubiquitinated proteins detected < 6 h: %d'%(len(Ubq_protein_df)))
Ubq_protein_df = Ubq_protein_df[Ubq_protein_df['KGG increase w MG132?'] == 'Y']
print('Number of UPS-ubiquitinated proteins detected < 6 h: %d'%(len(Ubq_protein_df)))
Ubq_protein_df = birthdating_labelled_1d_df[(birthdating_labelled_1d_df['Label #1'] == 'YU')]
print('Number of YU proteins detected < 6 h: %d'%(len(Ubq_protein_df)))
print('%d of which are membrane proteins according to UniProt'%(len(Ubq_protein_df.merge(mem_prot_df, on='Uniprot', how='inner'))))
Ubq_AF_ent_df = Ubq_protein_df.merge(AF_ent_data, on='Uniprot', how='inner')
print('%d have high quality AlphaFold structures'%(len(Ubq_AF_ent_df)))
print('Amongst high quality structure, %d have one or more native entanglements'%(len(Ubq_AF_ent_df[Ubq_AF_ent_df['Entanglement']==1])))
print('%d have covalent lassos'%(len(Ubq_AF_ent_df[pd.isna(Ubq_AF_ent_df['Entanglement'])])))
print('%d have knots'%(len(Ubq_AF_ent_df[Ubq_AF_ent_df['Knotted']==1])))

Total number of proteins detected: 6450
2194 are membrane proteins according to UniProt
2280 have high quality AlphaFold structures
Amongst high quality structure, 1651 have one or more native entanglements
47 have covalent lassos
15 have knots
Total number of ubiquitinated proteins detected < 6 h: 1384
Number of UPS-ubiquitinated proteins detected < 6 h: 1299
Number of YU proteins detected < 6 h: 906
341 of which are membrane proteins according to UniProt
507 have high quality AlphaFold structures
Amongst high quality structure, 432 have one or more native entanglements
3 have covalent lassos
1 have knots


In [8]:
df_0 = birthdating_AF_ent_df.merge(mem_prot_df, on='Uniprot', how='left')
print('%.4f%% (%d out of %d) of the mass spec observable globular proteins contain one or more native NCLEs.'%(len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y')]) / len(df_0[df_0['Membrane?']!='Y']) * 100 ,
                                                                                                               len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y')]),
                                                                                                               len(df_0[df_0['Membrane?']!='Y'])))
print('Of these, %.4f%% (%d out of %d) get ubiquitinated for proteasome degradation.'%(len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y') & (df_0['KGG increase w MG132?']=='Y')]) / len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y')]) * 100,
                                                                                       len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y') & (df_0['KGG increase w MG132?']=='Y')]),
                                                                                       len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y')])))
print('%.4f%% (%d out of %d) do not get ubiquitinated.'%(len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y') & (df_0['KGG peptides detected?'] == 'N')]) / len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y')]) * 100,
                                                         len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y') & (df_0['KGG peptides detected?'] == 'N')]),
                                                         len(df_0[(df_0['Entanglement']==1) & (df_0['Membrane?']!='Y')])))

73.6535% (1135 out of 1541) of the mass spec observable globular proteins contain one or more native NCLEs.
Of these, 49.0749% (557 out of 1135) get ubiquitinated for proteasome degradation.
43.7885% (497 out of 1135) do not get ubiquitinated.
